In [1]:
import pandas as pd
import os
import numpy as np
from params import ROOT_DIR
from pathlib import Path

In [2]:
data = pd.read_excel("../oos_data.xlsx")

In [3]:
data

,Patient,date of birth,gender,eye,OCT date,G,T,TS,NS,N,NI,TI,Glaucoma
0,mg1001,1980-12-20,NaN,OD,2021-08-28,104,71,135,138,85,124,148,0
1,mg1001,1980-12-20,NaN,OS,2021-08-28,108,68,124,164,101,124,132,0
2,mg1002,1948-06-26,1.0,OD,2021-09-02,98,72,165,107,63,126,146,0
3,mg1002,1948-06-26,1.0,OS,2021-09-02,87,57,135,109,75,109,93,1
4,mg1005,1953-09-04,0.0,OD,2021-09-02,91,65,139,87,72,111,142,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
112,mg1079,1974-01-22,NaN,OS,2022-05-11,94,63,95,165,78,94,140,1
113,mg1081,1957-09-27,1.0,OD,2022-06-01,38,47,40,54,18,51,45,1
114,mg1081,1957-09-27,1.0,OS,2022-06-01,97,61,101,123,77,144,158,0
115,mg1082,1953-09-25,0.0,OD,2022-06-01,105,77,146,132,75,135,153,0


In [4]:
now = pd.Timestamp('now')
data['date of birth'] = data['date of birth'].where(data['date of birth'] < now, data['date of birth'] -  np.timedelta64(100, 'Y'))   # 2
data['age'] = (now - data['date of birth']).astype('<m8[Y]')    # 3
data

,Patient,date of birth,gender,eye,OCT date,G,T,TS,NS,N,NI,TI,Glaucoma,age
0,mg1001,1980-12-20,NaN,OD,2021-08-28,104,71,135,138,85,124,148,0,41.0
1,mg1001,1980-12-20,NaN,OS,2021-08-28,108,68,124,164,101,124,132,0,41.0
2,mg1002,1948-06-26,1.0,OD,2021-09-02,98,72,165,107,63,126,146,0,74.0
3,mg1002,1948-06-26,1.0,OS,2021-09-02,87,57,135,109,75,109,93,1,74.0
4,mg1005,1953-09-04,0.0,OD,2021-09-02,91,65,139,87,72,111,142,0,69.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112,mg1079,1974-01-22,NaN,OS,2022-05-11,94,63,95,165,78,94,140,1,48.0
113,mg1081,1957-09-27,1.0,OD,2022-06-01,38,47,40,54,18,51,45,1,65.0
114,mg1081,1957-09-27,1.0,OS,2022-06-01,97,61,101,123,77,144,158,0,65.0
115,mg1082,1953-09-25,0.0,OD,2022-06-01,105,77,146,132,75,135,153,0,69.0


In [5]:
data.columns.values

array(['Patient', 'date of birth', 'gender', 'eye', 'OCT date', 'G', 'T',
       'TS', 'NS', 'N', 'NI', 'TI', 'Glaucoma', 'age'], dtype=object)

In [6]:
len(data[data["Glaucoma"] == 1])/len(data)

0.3247863247863248

In [7]:
patients_folder = os.listdir("../imgs2/")

In [8]:
not_have_folder = []
id_names = data["Patient"].values.tolist()
for patient in id_names:
    if patient.upper() in patients_folder:
        continue
    else:
        not_have_folder.append(patient)

In [9]:
not_have_folder

[]

In [10]:
percent_missing = data.isnull().sum() * 100 / len(data)
missing_value_df = pd.DataFrame({'column_name': data.columns,
                                 'percent_missing': percent_missing})
missing_value_df

,column_name,percent_missing
Patient,Patient,0.000000
date of birth,date of birth,0.000000
gender,gender,11.965812
eye,eye,0.000000
OCT date,OCT date,0.000000
G,G,0.000000
T,T,0.000000
TS,TS,0.000000
NS,NS,0.000000
N,N,0.000000


In [11]:
data.shape

(117, 14)

In [12]:
dict_data = {"Patient": [], "label": [], "photo_1": [], "photo_2": [], "eye_side": [], 
            "gender": [], "G": [], "T": [], "TS": [], "NS": [], "N": [], "NI": [], "TI": [], "age": []}
count = 0
for name, patient in data.iterrows():
    folder_imgs =  os.listdir("../imgs2/" + patient["Patient"].upper())
        
    
    dict_data["G"].append(patient["G"])
    dict_data["T"].append(patient["T"])
    dict_data["TS"].append(patient["TS"])
    dict_data["NS"].append(patient["NS"])
    dict_data["N"].append(patient["N"])
    dict_data["NI"].append(patient["NI"])
    dict_data["TI"].append(patient["TI"])
    dict_data["gender"].append(float(patient["gender"]))
    dict_data["age"].append(float(patient["age"]))
    dict_data["Patient"].append(patient["Patient"].upper())
    dict_data["label"].append(patient["Glaucoma"])

    if patient["eye"].upper() == "OD" or patient["eye"] is np.nan:
        dict_data["eye_side"].append("right")

        if "111111_Color_11MP_R_001.jpg" in list(folder_imgs):
            dict_data["photo_1"].append(Path("imgs/" + patient["Patient"].upper() + "/111111_Color_11MP_R_001.jpg"))
        else:
            dict_data["photo_1"].append(np.nan)

        if "111111_Color_11MP_R_002.jpg" in folder_imgs:
            dict_data["photo_2"].append(Path("imgs/" + patient["Patient"].upper() + "/111111_Color_11MP_R_002.jpg"))
        else:
            dict_data["photo_2"].append(np.nan) 

    elif patient["eye"].upper() == "OS" :

        dict_data["eye_side"].append("left")


        if "111111_Color_11MP_L_001.jpg" in folder_imgs:
            dict_data["photo_1"].append(Path("imgs/" + patient["Patient"].upper() + "/111111_Color_11MP_L_001.jpg"))
        else:
            dict_data["photo_1"].append(np.nan)
        if "111111_Color_11MP_L_002.jpg" in folder_imgs:
            dict_data["photo_2"].append(Path("imgs/" + patient["Patient"].upper() + "/111111_Color_11MP_L_002.jpg"))
        else:
            dict_data["photo_2"].append(np.nan)


        
   

In [13]:
dict_data.keys()

dict_keys(['Patient', 'label', 'photo_1', 'photo_2', 'eye_side', 'gender', 'G', 'T', 'TS', 'NS', 'N', 'NI', 'TI', 'age'])

In [14]:
path_df = pd.DataFrame.from_dict(dict_data)
path_df

,Patient,label,photo_1,photo_2,eye_side,gender,G,T,TS,NS,N,NI,TI,age
0,MG1001,0,imgs/MG1001/111111_Color_11MP_R_001.jpg,imgs/MG1001/111111_Color_11MP_R_002.jpg,right,NaN,104,71,135,138,85,124,148,41.0
1,MG1001,0,imgs/MG1001/111111_Color_11MP_L_001.jpg,imgs/MG1001/111111_Color_11MP_L_002.jpg,left,NaN,108,68,124,164,101,124,132,41.0
2,MG1002,0,imgs/MG1002/111111_Color_11MP_R_001.jpg,imgs/MG1002/111111_Color_11MP_R_002.jpg,right,1.0,98,72,165,107,63,126,146,74.0
3,MG1002,1,imgs/MG1002/111111_Color_11MP_L_001.jpg,imgs/MG1002/111111_Color_11MP_L_002.jpg,left,1.0,87,57,135,109,75,109,93,74.0
4,MG1005,0,imgs/MG1005/111111_Color_11MP_R_001.jpg,imgs/MG1005/111111_Color_11MP_R_002.jpg,right,0.0,91,65,139,87,72,111,142,69.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112,MG1079,1,imgs/MG1079/111111_Color_11MP_L_001.jpg,imgs/MG1079/111111_Color_11MP_L_002.jpg,left,NaN,94,63,95,165,78,94,140,48.0
113,MG1081,1,imgs/MG1081/111111_Color_11MP_R_001.jpg,imgs/MG1081/111111_Color_11MP_R_002.jpg,right,1.0,38,47,40,54,18,51,45,65.0
114,MG1081,0,imgs/MG1081/111111_Color_11MP_L_001.jpg,imgs/MG1081/111111_Color_11MP_L_002.jpg,left,1.0,97,61,101,123,77,144,158,65.0
115,MG1082,0,imgs/MG1082/111111_Color_11MP_R_001.jpg,imgs/MG1082/111111_Color_11MP_R_002.jpg,right,0.0,105,77,146,132,75,135,153,69.0


In [17]:
percent_missing = path_df.isnull().sum() * 100 / len(path_df)
missing_value_df = pd.DataFrame({'column_name': path_df.columns,
                                 'percent_missing': percent_missing})
missing_value_df

,column_name,percent_missing
Patient,Patient,0.000000
label,label,0.000000
photo_1,photo_1,2.564103
photo_2,photo_2,2.564103
eye_side,eye_side,0.000000
gender,gender,11.965812
G,G,0.000000
T,T,0.000000
TS,TS,0.000000
NS,NS,0.000000


In [18]:
list_of_numerical_f = ["T","TS","NS","N","NI","TI"]

for idx,i in enumerate(list_of_numerical_f):
    if idx + 1 == len(list_of_numerical_f):
        break
    for edx, j in enumerate(list_of_numerical_f[idx+1:]):
        column_name = i + "/" + j
        path_df[column_name] = path_df[i]/path_df[j]


In [19]:
completed_df = path_df.dropna(axis=0, subset=["photo_1","photo_2","gender"])
len(completed_df[completed_df["label"] == 1])/len(completed_df)

0.33

In [20]:
completed_df.shape

(100, 29)

In [21]:
completed_df.to_csv("../data_oos.csv",index=False)